# Secure distributed Computation Final Project
## Luke Holmes and Jake Pappas

We are going to implement linear regression inference on secret inputs to provide disease progression inferences to clients securely. We will use a 1-server, n-clients architecture, using OT to generate multiplication triples. Our dataset is the scikit-learn diabetes dataset.

In [29]:
# Useful imports and utility functions
import pychor
import galois
import pandas as pd
import sklearn
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression

server = pychor.Party('server')
user = pychor.Party('user')

import numpy as np

GF = galois.GF(2**61-1)

@pychor.local_function
def share(x):
    s1 = GF.Random()
    s2 = GF(x) - s1
    return s1, s2

In [30]:
# Load as a pandas dataframe
diabetes_df = server.constant(load_diabetes(as_frame=True))
df = server.constant(diabetes_df.val.frame)
model = LinearRegression()
model.fit(df.val.drop(columns='target'), df.val['target'])
scaled_coefs = model.coef_ * 10**8

print(f"Coefficents:{scaled_coefs}")
print(f"Intercept:{model.intercept_}")

Coefficents:[-1.00098663e+09 -2.39815644e+10  5.19845920e+10  3.24384646e+10
 -7.92175639e+10  4.76739021e+10  1.01043268e+10  1.77063238e+10
  7.51273700e+10  6.76266922e+09]
Intercept:152.13348416289597


In [31]:
from dataclasses import dataclass

multiplication_triples = []

@dataclass
class SecInt:
    # s1 is p1's share of the value, and s2 is p2's share
    s1: galois.GF
    s2: galois.GF

    @classmethod
    def input(cls, val):
        """Secret share an input: p1 holds s1, and p2 holds s2"""
        s1, s2 = share(val).untup(2)
        if p1 in val.parties:
            s2.send(p1, p2)
            return SecInt(s1, s2)
        else:
            s1.send(p2, p1)
            return SecInt(s1, s2)

    def __add__(x, y):
        """Add two SecInt objects using local addition of shares"""
        return SecInt(x.s1 + y.s1, x.s2 + y.s2)

    def __mul__(x, y):
        """Multiply two SecInt objects using a triple"""
        triple = multiplication_triples.pop()
        r1, r2 = protocol_mult((x.s1, x.s2), (y.s1, y.s2), triple)
        return SecInt(r1, r2)

    def reveal(self):
        """Reveal the secret value by broadcast and reconstruction"""
        self.s1.send(p1, p2)
        self.s2.send(p2, p1)
        return self.s1 + self.s2

def functionality_gen_triple():
    Fgen = pychor.Party('Fgen')

    def deal_shares(x):
        s1, s2 = share(x).untup(2)
        s1.send(Fgen, p1)
        s2.send(Fgen, p2)
        return s1, s2

    # Step 1: generate a, b, c
    a = Fgen.constant(GF.Random())
    b = Fgen.constant(GF.Random())
    c = a * b

    # Step 2: secret share a, b, c
    a1, a2 = deal_shares(a)
    b1, b2 = deal_shares(b)
    c1, c2 = deal_shares(c)
    return (a1, a2), (b1, b2), (c1, c2)

def protocol_mult(x, y, triple):
    x1, x2 = x
    y1, y2 = y
    (a1, a2), (b1, b2), (c1, c2) = triple

    # Step 1. P1 computes d_1 = x_1 - a_1 and sends the result to P2
    d1 = x1 - a1
    d1.send(p1, p2)

    # Step 2. P2 computes d_2 = x_2 - a_2 and sends the result to P1
    d2 = x2 - a2
    d2.send(p2, p1)

    # Step 3: P1 and P2 both compute $d = d_1 + d_2 = x - a$
    d = d1 + d2

    # Step 4. P1 computes e_1 = y_1 - b_1 and sends the result to P2
    e1 = y1 - b1
    e1.send(p1, p2)

    # Step 5. P2 computes e_2 = y_2 - b_2 and sends the result to P1    
    e2 = y2 - b2
    e2.send(p2, p1)

    # Step 6. P1 and P2 both compute $e = e_1 + e_2 = y - b$
    e = e1 + e2

    # Step 7. P1 computes r_1 = d*e + d*b_1 + e*a_1 + c_1
    r_1 = d * e + d * b1 + e * a1 + c1

    # Step 8. P2 computes r_2 = 0 + d*b_2 + e*a_2 + c_2
    r_2 = d * b2 + e * a2 + c2

    return r_1, r_2

def gen_triples(n):
    for _ in range(n):
        multiplication_triples.append(functionality_gen_triple())